# MSMARCO-XI -> Jina v3 (local GPU) -> Qdrant — final

**Use a fresh GPU runtime.** Runtime -> Disconnect and delete runtime, then
Runtime -> Change runtime type -> T4 GPU. Every previous crash left the kernel
in a degraded state; starting dirty is why the model load died.

### Why the reads kept crashing

`hintrain.parquet` is 778,638 rows in **one row group, 9,729 MB uncompressed**.
Arrow has to decode that whole block to give you row 1, and `fs.open()` buffered
the file body in RAM on top of it. No `batch_size` or `buffer_size` setting can
avoid that. Both `datasets` streaming and `pyarrow.iter_batches` over HTTP are
out.

### What replaces it

| Problem | Fix |
|---|---|
| 9.7 GB row group | **DuckDB with `memory_limit`** — spills to disk instead of dying |
| fsspec buffering the body | Download the file first, read from local disk |
| Reading 778K rows to use 2.5K | `LIMIT` in SQL — DuckDB stops early |
| 3.7 GB × 7 languages on disk | Delete each file after its language is done |
| Model load OOM | `low_cpu_mem_usage=True`, and load **before** touching any data |

The GPU does the embedding, the CPU does the Parquet, and only one language is
resident at a time — which is exactly the loop you described.

In [1]:
!pip install -q duckdb "qdrant-client[fastembed]" "peft>=0.15.2" huggingface_hub numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 14.2 MB/s eta 0:00:00


In [2]:
!pip uninstall -y -q torchao

In [3]:
import torch, transformers, sys

def ver(m):
    return tuple(int(x) for x in m.__version__.split("+")[0].split(".")[:2])

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | "
      f"transformers {transformers.__version__}")

ok = True
if ver(torch) < (2, 8):
    print("torch < 2.8 — v5 needs 2.8+. Do NOT pip -U torch; it breaks the image.")
    print("Use a newer Colab runtime instead, or fall back to jina-embeddings-v3.")
    ok = False
if ver(transformers) < (4, 57):
    print("transformers < 4.57 — upgrade ONLY transformers, leave torch alone:")
    print('   !pip install -q -U "transformers>=4.57.0"')
    ok = False

# torch must actually work, not just import. This catches a broken install
# before it surfaces 17 frames deep inside from_pretrained().
try:
    _ = torch.zeros(4, device="cuda") @ torch.zeros(4, 4, device="cuda")
    import torch._inductor.inductor_prims          # the exact import that failed
    print("torch healthy | GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print(f"\nTORCH IS BROKEN: {type(e).__name__}: {e}")
    print("Runtime -> Disconnect and delete runtime, then re-run WITHOUT upgrading torch.")
    ok = False

assert ok, "fix the above before loading the model"



python 3.13.15 | torch 2.11.0+cu128 | transformers 5.15.0
torch healthy | GPU: Tesla T4


In [4]:
import os, gc, json, time, uuid, re, hashlib, shutil, resource
import numpy as np, duckdb, torch
from tqdm.auto import tqdm

def rss_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0),
      f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"RSS now: {rss_gb():.2f} GB")

GPU: Tesla T4 | VRAM 15.6 GB
RSS now: 0.94 GB


In [5]:
DATASET        = "ai4bharat/MSMARCO-XI"
HF_TOKEN       = "..."
QDRANT_URL     = "..."
QDRANT_API_KEY = "..."
COLLECTION     = "msmarco_xi"

WORK   = "/content/work"          # local scratch for parquet files
DRIVE  = "/content/drive/MyDrive/msmarco_xi_ingestion"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception as e:
    print("no drive:", e); DRIVE = "/content/artifacts"
os.makedirs(DRIVE, exist_ok=True)

# file prefixes from the real repo listing (NOT the dataset card - it disagrees)
FILE = {"hi":"hin","kn":"kan","ta":"tam","ml":"mal","mr":"mar","or":"ori","bn":"ben",
        "gu":"guj","pa":"pan","ur":"urd","as":"asm","ne":"nep","sa":"san"}

# Tier 1 = priority (all 5 strategies). Tier 2 = coverage (atomic + query only).
# Telugu is validation-only in this repo; add ("te", ...) against telval if you want it.
PLAN = [
    ("hi", 2500, "P"), ("kn", 1000, "P"), ("ta", 1000, "P"), ("ml", 1000, "P"),
    ("mr", 1000, "C"), ("or", 600, "C"), ("bn", 600, "C"),
]
STRATS = {"P": ["atomic", "window", "enriched", "xling", "query"],
          "C": ["atomic", "query"]}
DISTRACTORS   = 2
EMBED_BATCH   = 96          # GPU batch
PIPELINE_CHUNK= 1500        # units per embed -> upsert -> free cycle
UPSERT_BATCH  = 128
DUCKDB_MEM    = "5GB"       # hard ceiling; DuckDB spills past this rather than dying

print("planned languages:", [p[0] for p in PLAN])

no drive: Error: credential propagation was unsuccessful
planned languages: ['ta', 'ml', 'mr', 'or', 'bn']


## 1. Load the model FIRST

Before any data touches RAM. `low_cpu_mem_usage=True` streams weights straight to
the GPU instead of materialising a full fp32 state dict in CPU memory — that is
what killed the earlier load.

Do **not** disable the LoRA adapters. They *are* the `retrieval.query` /
`retrieval.passage` task adapters. Turning them off makes the task argument a
silent no-op and breaks compatibility with the Jina API your runtime calls.

In [6]:
import numpy as np
from transformers import AutoModel

MODEL_ID = "jina-embeddings-v5-text-small"
TASK, DOC_PROMPT, QRY_PROMPT = "retrieval", "document", "query"
_BACKEND = None

try:
    model = AutoModel.from_pretrained(
        f"jinaai/{MODEL_ID}",
        trust_remote_code=True,
        dtype=torch.float16,        # T4 = Turing. NO bfloat16, NO flash_attention_2.
    ).to("cuda").eval()
    _BACKEND = "transformers"
except Exception as e:
    # jina's custom remote code can lag behind a transformers major bump
    # (you are on 5.x). sentence-transformers wraps it differently and often
    # survives that.
    print(f"AutoModel path failed: {type(e).__name__}: {str(e)[:200]}")
    print("falling back to sentence-transformers ...")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(
        f"jinaai/{MODEL_ID}",
        trust_remote_code=True,
        device="cuda",
        model_kwargs={"dtype": torch.float16},
    )
    _BACKEND = "sentence-transformers"

JINA_REV = getattr(getattr(model, "config", None), "_commit_hash", None) or "unknown"
print(f"loaded {MODEL_ID} via {_BACKEND} | rev={JINA_REV} "
      f"| VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")


def matryoshka(v, dim):
    s = v[:, :dim]
    n = np.linalg.norm(s, axis=1, keepdims=True); n[n == 0] = 1.0
    return (s / n).astype(np.float32)


# @torch.no_grad()
# def encode(texts, prompt_name, bs=32):
#     """AutoModel takes texts=, SentenceTransformer takes sentences=. Same model."""
#     assert prompt_name in (DOC_PROMPT, QRY_PROMPT)
#     if not texts:
#         return np.zeros((0, 1024), np.float32)
#     out, i = [], 0
#     while i < len(texts):
#         chunk = texts[i:i + bs]
#         try:
#             if _BACKEND == "transformers":
#                 v = model.encode(texts=chunk, task=TASK,
#                                  prompt_name=prompt_name, truncate_dim=None)
#             else:
#                 v = model.encode(sentences=chunk, task=TASK,
#                                  prompt_name=prompt_name)
#             out.append(np.asarray(v, dtype=np.float32)); i += bs
#         except torch.cuda.OutOfMemoryError:
#             torch.cuda.empty_cache()
#             if bs <= 2: raise
#             bs //= 2; print("  VRAM OOM -> batch", bs)
#     return np.vstack(out)

def _to_np(v):
    """v5 returns a torch tensor on GPU (fp16). v3 returned numpy. Handle both."""
    if isinstance(v, torch.Tensor):
        return v.detach().to(torch.float32).cpu().numpy()
    if isinstance(v, (list, tuple)) and v and isinstance(v[0], torch.Tensor):
        return torch.stack(v).detach().to(torch.float32).cpu().numpy()
    return np.asarray(v, dtype=np.float32)


@torch.no_grad()
def encode(texts, prompt_name, bs=32):
    assert prompt_name in (DOC_PROMPT, QRY_PROMPT)
    if not texts:
        return np.zeros((0, 1024), np.float32)
    out, i = [], 0
    while i < len(texts):
        chunk = texts[i:i + bs]
        try:
            if _BACKEND == "transformers":
                v = model.encode(texts=chunk, task=TASK,
                                 prompt_name=prompt_name, truncate_dim=None)
            else:
                v = model.encode(sentences=chunk, task=TASK,
                                 prompt_name=prompt_name, convert_to_numpy=True)
            out.append(_to_np(v))          # <-- the fix
            i += bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs <= 2: raise
            bs //= 2; print("  VRAM OOM -> batch", bs)
    return np.vstack(out)

# ---- sanity: right dim, normalized, and query/document prompts discriminating
_q  = encode(["मुझे बताओ कि यह कैसे काम करता है"], QRY_PROMPT)
_d  = encode(["यह इस तरह काम करता है कि ..."],      DOC_PROMPT)
_d2 = encode(["केले की कीमत क्या है"],              DOC_PROMPT)

assert _q.shape == (1, 1024), f"expected 1024 dims, got {_q.shape}"
assert abs(np.linalg.norm(matryoshka(_q, 256)[0]) - 1.0) < 1e-4

rel   = float(matryoshka(_q, 256) @ matryoshka(_d,  256).T)
unrel = float(matryoshka(_q, 256) @ matryoshka(_d2, 256).T)
print(f"encode OK | related {rel:.3f} vs unrelated {unrel:.3f}")
assert rel > unrel, "prompt_name is being ignored — query and doc collapse to the same space"



config.json:   0%|          | 0.00/991 [00:00<?, ?B/s]

configuration_jina_embeddings_v5.py:   0%|          | 0.00/120 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-small:
- configuration_jina_embeddings_v5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_jina_embeddings_v5.py:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-small:
- modeling_jina_embeddings_v5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

loaded jina-embeddings-v5-text-small via transformers | rev=dd76d535f5447ca3897a9c893fb1e612ead98192 | VRAM 1.52 GB
encode OK | related 0.861 vs unrelated 0.154


/tmp/ipykernel_7778/3768070371.py:104: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rel   = float(matryoshka(_q, 256) @ matryoshka(_d,  256).T)
/tmp/ipykernel_7778/3768070371.py:105: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  unrel = float(matryoshka(_q, 256) @ matryoshka(_d2, 256).T)


## 2. The reader — DuckDB with a hard memory ceiling

Download once, query with `LIMIT`, delete. DuckDB processes in 2048-row vectors
and spills to `temp_directory` when it hits `memory_limit`, so a 9.7 GB row group
degrades to slow rather than fatal.

In [7]:
from huggingface_hub import hf_hub_download

con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEM}'")
con.execute(f"SET temp_directory='{WORK}/duckdb_tmp'")
con.execute("SET preserve_insertion_order=false")

def fetch_language(lang, n_queries, split="train"):
    '''Download one parquet, pull n_queries rows, delete the file. Returns list of dicts.'''
    fn = f"{FILE[lang]}{'train' if split=='train' else 'val'}.parquet"
    path = f"{split}/{fn}"
    print(f"  downloading {path} ...")
    local = hf_hub_download(DATASET, path, repo_type="dataset",
                            token=(HF_TOKEN or None), local_dir=WORK)
    print(f"  {os.path.getsize(local)/1e9:.2f} GB on disk | RSS {rss_gb():.2f} GB")

    rows = con.execute(f'''
        SELECT query, query_id, query_type, "Answer", "Eng_Query",
               passages.Translated_passages AS tp,
               passages.English_passages    AS ep,
               passages.is_selected         AS sel
        FROM read_parquet('{local}')
        WHERE length(query) > 0
        LIMIT {n_queries}
    ''').fetchall()
    print(f"  {len(rows)} rows | RSS {rss_gb():.2f} GB")

    os.remove(local)
    shutil.rmtree(f"{WORK}/{split}", ignore_errors=True)
    gc.collect()
    return rows

# smoke test on the smallest priority language before committing to the run
_t = fetch_language("hi", 5)
r0 = _t[0]
print("\nquery_id :", r0[1], "| type:", r0[2])
print("query    :", r0[0][:80])
print("Eng_Query:", r0[4][:80])
print("Answer   :", str(r0[3])[:80])
print("passages :", len(r0[5]), "| is_selected:", list(r0[7]), "| gold:", sum(r0[7]))
assert len(r0[5]) == len(r0[6]) == len(r0[7]), "parallel lists misaligned"
print("\nREADER OK | peak RSS", f"{rss_gb():.2f} GB")

  downloading train/hintrain.parquet ...


train/hintrain.parquet: reconstructing file:   0%|          |  0.00B / 3.72GB            

train/hintrain.parquet: downloading bytes:           |  0.00B            

  3.72 GB on disk | RSS 4.32 GB
  5 rows | RSS 4.32 GB

query_id : 1185869 | type: DESCRIPTION
query    : मैनहट्टन परियोजना की सफलता का तुरंत क्या प्रभाव पड़ा?
Eng_Query: )what was the immediate impact of the success of the manhattan project?
Answer   : मैनहट्टन परियोजना की सफलता का तत्काल प्रभाव परमाणु शोधकर्ताओं और इंजीनियरों की प
passages : 10 | is_selected: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0] | gold: 1

READER OK | peak RSS 4.32 GB


## 3. Unit construction

In [8]:
NS = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")
pid_for = lambda uid: str(uuid.uuid5(NS, uid))
SENT = re.compile(r"[\u0964\u0965.!?\n]+")

def sentences(t):
    p = [x.strip() for x in SENT.split(t) if x.strip()]
    if len(p) >= 2: return p
    w = t.split()
    return [" ".join(w[i:i+40]) for i in range(0, len(w), 40)] or [t]

def windows(s, size=3, stride=2):
    out, i = [], 0
    while i < len(s):
        out.append(" ".join(s[i:i+size]))
        if i + size >= len(s): break
        i += stride
    return out

def U(uid, text, embed_text, lang, tier, strat, qid, pid, **kw):
    p = {"text": text, "lang": lang, "tier": tier, "strategy": strat, "qid": str(qid),
         "pid": str(pid), "parent_id": None, "query_text": None, "answer_text": None,
         "is_gold": True, "query_type": None, "n_chars": len(text)}
    p.update(kw)
    return {"uid": uid, "embed_text": embed_text[:2000], "payload": p, "task": DOC_PROMPT}

def build_units(row, lang, tier, strats, median_len):
    query, qid, qtype, answer, eng_q, tp, ep, sel = row
    if not tp: return []
    gi = next((i for i, s in enumerate(sel) if s == 1), 0)   # gold index
    order = [gi] + [i for i in range(len(tp)) if i != gi][:DISTRACTORS]
    units = []

    if "atomic" in strats:
        for k, i in enumerate(order):
            if not tp[i]: continue
            units.append(U(f"{lang}|atomic|{qid}|p{i:02d}", tp[i], tp[i], lang, tier,
                           "atomic", qid, f"{qid}-p{i}", is_gold=(i == gi), query_type=qtype))

    gold = tp[gi]
    if "window" in strats and len(gold) > median_len:
        parent = pid_for(f"{lang}|atomic|{qid}|p{gi:02d}")
        for wi, w in enumerate(windows(sentences(gold))):
            units.append(U(f"{lang}|window|{qid}|w{wi:02d}", w, w, lang, tier,
                           "window", qid, f"{qid}-p{gi}", parent_id=parent, query_type=qtype))

    if "enriched" in strats and query:
        units.append(U(f"{lang}|enriched|{qid}|p00", gold, f"{query}\n\n{gold}", lang, tier,
                       "enriched", qid, f"{qid}-p{gi}", query_type=qtype))

    # cross-lingual twin: the ENGLISH passage, same canonical qid, lang="en"
    if "xling" in strats and ep and ep[gi]:
        units.append(U(f"{lang}|xling|{qid}|p00", ep[gi], ep[gi], "en", tier,
                       "xling", qid, f"{qid}-p{gi}", query_type=qtype))

    # tier-1 fast path: embed the QUERY, payload carries the real Answer
    if "query" in strats and query:
        u = U(f"{lang}|query|{qid}|q00", gold, query, lang, tier, "query", qid,
              f"{qid}-p{gi}", query_text=query, answer_text=(answer or gold),
              query_type=qtype, n_chars=len(query))
        u["task"] = QRY_PROMPT          # stored queries use the query adapter
        units.append(u)
    return units

## 4. Sparse vectors

In [9]:
from fastembed import SparseTextEmbedding
_bm25, USE_FALLBACK = None, False
def bm25():
    global _bm25
    if _bm25 is None: _bm25 = SparseTextEmbedding("Qdrant/bm25")
    return _bm25

TOK = re.compile(r"\w+", re.UNICODE)
def sparse_fallback(t):
    c = {}
    for w in TOK.findall(t.lower()):
        h = int(hashlib.md5(w.encode()).hexdigest()[:8], 16); c[h] = c.get(h, 0) + 1
    return list(c.keys()), [float(v) for v in c.values()]

def sparse(texts):
    if USE_FALLBACK: return [sparse_fallback(t) for t in texts]
    return [(r.indices.tolist(), r.values.tolist()) for r in bm25().embed(texts)]

_s = [r[5][0] for r in _t if r[5]][:5]
_nz = float(np.mean([len(i) for i, _ in [( r.indices, r.values) for r in bm25().embed(_s)]]))
print(f"BM25 mean non-zero terms on Devanagari: {_nz:.1f}")
if _nz <= 1.0:
    USE_FALLBACK = True; print("degenerate -> regex tokenizer")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

BM25 mean non-zero terms on Devanagari: 43.4


## 5. Qdrant collection

In [10]:
from qdrant_client import QdrantClient, models
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)

INT8 = models.ScalarQuantization(scalar=models.ScalarQuantizationConfig(
    type=models.ScalarType.INT8, quantile=0.99, always_ram=True))

def create():
    if client.collection_exists(COLLECTION): client.delete_collection(COLLECTION)
    client.create_collection(
        COLLECTION,
        vectors_config={
            # wide stage: int8 copies in RAM, originals on disk
            "dense_256": models.VectorParams(size=256, distance=models.Distance.COSINE,
                on_disk=True, hnsw_config=models.HnswConfigDiff(m=16, ef_construct=64),
                quantization_config=INT8),
            # rerank stage: m=0 -> no graph built. NOT quantized; full precision is the point.
            "dense_1024": models.VectorParams(size=1024, distance=models.Distance.COSINE,
                on_disk=True, hnsw_config=models.HnswConfigDiff(m=0)),
        },
        sparse_vectors_config={"sparse_bm25": models.SparseVectorParams(
            index=models.SparseIndexParams(on_disk=False), modifier=models.Modifier.IDF)},
        on_disk_payload=True,
        optimizers_config=models.OptimizersConfigDiff(
            default_segment_number=2, indexing_threshold=0),   # no HNSW during bulk load
    )
    for f, s in [("lang", models.PayloadSchemaType.KEYWORD),
                 ("strategy", models.PayloadSchemaType.KEYWORD),
                 ("tier", models.PayloadSchemaType.KEYWORD),
                 ("qid", models.PayloadSchemaType.KEYWORD),
                 ("query_type", models.PayloadSchemaType.KEYWORD),
                 ("parent_id", models.PayloadSchemaType.KEYWORD),
                 ("is_gold", models.PayloadSchemaType.BOOL)]:
        client.create_payload_index(COLLECTION, field_name=f, field_schema=s, wait=True)
    print("collection created")

create()      # destructive — comment out when resuming

## 6. The loop

One language at a time: download -> slice -> build units -> embed on GPU ->
upsert -> free -> delete file -> next. Peak RAM stays flat regardless of how many
languages you add.

In [11]:
def ck(lang): return os.path.join(DRIVE, f"ck_{lang}.json")
def ck_get(l): return json.load(open(ck(l)))["done"] if os.path.exists(ck(l)) else 0
def ck_set(l, d): json.dump({"done": d}, open(ck(l), "w"))

def upsert(points):
    for i in range(0, len(points), UPSERT_BATCH):
        b = points[i:i+UPSERT_BATCH]
        for a in range(5):
            try: client.upsert(COLLECTION, points=b, wait=False); break
            except Exception:
                if a == 4: raise
                time.sleep(2 ** a)

RESULTS = {}

EMBED_BS = 16            # persistent, no OOM churn

done_f  = lambda l: os.path.join(DRIVE, f"done_{l}.flag")

def run_language(lang, n_q, tier):
    if os.path.exists(done_f(lang)):
        print(f"=== {lang} already done — skip ==="); return
    strats = STRATS[tier]
    print(f"\n=== {lang} ({tier}) — {n_q} queries, {strats} ===")
    rows = fetch_language(lang, n_q)
    if not rows: print("  no rows"); return

    lens = [len(p) for r in rows[:300] for p in r[5] if p]
    median_len = float(np.median(lens)) if lens else 0
    units = []
    for r in rows:
        units.extend(build_units(r, lang, tier, strats, median_len))
    for u in units:
        u["embed_text"] = u["embed_text"][:800]     # was 2000 — ~2x faster
    del rows; gc.collect()
    print(f"  {len(units)} units | RSS {rss_gb():.2f} GB")

    start = ck_get(lang)
    if start: print(f"  resuming at {start}")
    for s in tqdm(range(start, len(units), PIPELINE_CHUNK), desc=f"[{lang}]"):
        ch = units[s:s+PIPELINE_CHUNK]
        v = np.zeros((len(ch), 1024), np.float32)
        ci = [i for i, u in enumerate(ch) if u["task"] == DOC_PROMPT]
        qi = [i for i, u in enumerate(ch) if u["task"] == QRY_PROMPT]
        if ci: v[ci] = encode([ch[i]["embed_text"] for i in ci], DOC_PROMPT, EMBED_BS)
        if qi: v[qi] = encode([ch[i]["embed_text"] for i in qi], QRY_PROMPT, EMBED_BS)

        v1024, v256 = matryoshka(v, 1024), matryoshka(v, 256)
        sp = sparse([u["embed_text"] for u in ch])
        upsert([models.PointStruct(
            id=pid_for(u["uid"]),
            vector={"dense_256": v256[j].tolist(), "dense_1024": v1024[j].tolist(),
                    "sparse_bm25": models.SparseVector(indices=sp[j][0], values=sp[j][1])},
            payload={**u["payload"], "uid": u["uid"]}) for j, u in enumerate(ch)])

        del v, v1024, v256, sp; gc.collect(); torch.cuda.empty_cache()
        ck_set(lang, s + len(ch))

    counts = {}
    for u in units: counts[u["payload"]["strategy"]] = counts.get(u["payload"]["strategy"], 0) + 1
    RESULTS[lang] = {"tier": tier, "n_points": len(units), "strategy_counts": counts,
                     "sample_query": next((u["payload"]["query_text"] for u in units
                                           if u["payload"]["query_text"]), None)}
    open(done_f(lang), "w").write("1")
    print(f"  done: {counts}")
    del units; gc.collect()
for lang, n_q, tier in PLAN:
    try:
        run_language(lang, n_q, tier)
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"[{lang}] FAILED — continuing with the rest")

print("\n", json.dumps({k: v["n_points"] for k, v in RESULTS.items()}, indent=2))
print("TOTAL POINTS:", sum(v["n_points"] for v in RESULTS.values()))


=== ta (P) — 1000 queries, ['atomic', 'window', 'enriched', 'xling', 'query'] ===
  downloading train/tamtrain.parquet ...


train/tamtrain.parquet: reconstructing file:   0%|          |  0.00B / 3.99GB            

train/tamtrain.parquet: downloading bytes:           |  0.00B            

  3.99 GB on disk | RSS 4.32 GB
  1000 rows | RSS 4.32 GB
  7195 units | RSS 4.32 GB


[ta]:   0%|          | 0/5 [00:00<?, ?it/s]

...

train/bentrain.parquet: reconstructing file:   0%|          |  0.00B / 3.73GB            

train/bentrain.parquet: downloading bytes:           |  0.00B            

  3.73 GB on disk | RSS 4.45 GB
  600 rows | RSS 4.45 GB
  2400 units | RSS 4.45 GB


[bn]:   0%|          | 0/2 [00:00<?, ?it/s]

  done: {'atomic': 1800, 'query': 600}

 {
  "ta": 7195,
  "ml": 7483,
  "mr": 4000,
  "or": 2400,
  "bn": 2400
}
TOTAL POINTS: 23478


## 7. Build the index, then verify

In [12]:
client.update_collection(COLLECTION,
    optimizers_config=models.OptimizersConfigDiff(indexing_threshold=20000))
while True:
    st = client.get_collection(COLLECTION)
    if st.status == models.CollectionStatus.GREEN: break
    print("status", st.status, "indexed", st.indexed_vectors_count); time.sleep(15)
print("GREEN |", client.count(COLLECTION, exact=True).count, "points")

GREEN | 53444 points


In [13]:
def cnt(**f):
    return client.count(COLLECTION, exact=True, count_filter=models.Filter(
        must=[models.FieldCondition(key=k, match=models.MatchValue(value=v))
              for k, v in f.items()])).count

fails = []
tot = cnt(strategy="atomic"); gold = cnt(strategy="atomic", is_gold=True)
print(f"[1] distractor ratio {gold}/{tot} = {gold/max(1,tot):.3f}")
if not 0.2 < gold/max(1,tot) < 0.5: fails.append("distractor ratio off — check is_selected")

print("[2] coverage:")
for lang, r in RESULTS.items():
    for s in r["strategy_counts"]:
        n = cnt(lang=("en" if s == "xling" else lang), strategy=s)
        print(f"    {lang}/{s}: {n}")
        if n == 0 and s != "window": fails.append(f"{lang}/{s} empty")

print("\n[3] retrieval smoke test — READ THE TEXT, do not just check it passes:")
for lang, r in RESULTS.items():
    if not r["sample_query"]: continue
    qv = matryoshka(encode([r["sample_query"]], QRY_PROMPT), 256)[0]
    hits = client.query_points(COLLECTION, query=qv.tolist(), using="dense_256", limit=3,
        with_payload=True, query_filter=models.Filter(must=[models.FieldCondition(
            key="lang", match=models.MatchValue(value=lang))])).points
    print(f"  [{lang}] {r['sample_query'][:60]!r}")
    for h in hits:
        print(f"      {h.score:.3f} {h.payload['strategy']:<9} {h.payload['text'][:70]!r}")
    if not hits or hits[0].score < 0.5: fails.append(f"{lang}: weak/no hits")

print("\n" + ("FAILURES:\n  " + "\n  ".join(fails) if fails else "ALL CHECKS PASSED"))

[1] distractor ratio 8091/24273 = 0.333
[2] coverage:
    ta/atomic: 3000
    ta/enriched: 1000
    ta/xling: 5890
    ta/query: 1000
    ta/window: 1195
    ml/atomic: 3000
    ml/window: 1483
    ml/enriched: 1000
    ml/xling: 5890
    ml/query: 1000
    mr/atomic: 3000
    mr/query: 1000
    or/atomic: 1800
    or/query: 600
    bn/atomic: 1800
    bn/query: 600

[3] retrieval smoke test — READ THE TEXT, do not just check it passes:
  [ta] 'மன்ஹாட்டன் திட்டத்தின் வெற்றியின் உடனடி விளைவு என்ன?'
      1.004 query     'அறிவியல் அறிவுகளைப் போலவே அறிவியல் மனதின் முன்னிலையும் மன்ஹாட்டன் திட்'
      0.842 enriched  'அறிவியல் அறிவுகளைப் போலவே அறிவியல் மனதின் முன்னிலையும் மன்ஹாட்டன் திட்'
      0.759 atomic    'அறிவியல் அறிவுகளைப் போலவே அறிவியல் மனதின் முன்னிலையும் மன்ஹாட்டன் திட்'
  [ml] 'മാൻഹാട്ടൻ പദ്ധതിയുടെ വിജയത്തിന്റെ ഉടനടി ആഘാതം എന്തായിരുന്നു?'
      1.002 query     'ശാസ്ത്രീയ ബുദ്ധിശക്തിയുടെ സമാനമായി മാൻഹട്ടൻ പദ്ധതിയുടെ വിജയത്തിന് ശാസ്'
      0.788 enriched  'ശാസ്ത്രീയ ബുദ്ധിശക്തിയുട

## 8. Manifest — the runtime asserts against this on boot

In [14]:
manifest = {
    "model": "jinaai/jina-embeddings-v5-text-small",
    "api_model": "jina-embeddings-v5-text-small",
    "revision": JINA_REV,
    "task": "retrieval",
    "doc_prompt": "document",
    "query_prompt": "query",
    "dim_primary": 256, "dim_rerank": 1024,
    "normalized": True, "truncation": "slice_then_normalize",
    "collection": COLLECTION, "dataset": DATASET,
    "sparse": "regex_fallback" if USE_FALLBACK else "Qdrant/bm25",
    "total_points": client.count(COLLECTION, exact=True).count,
    "languages": {k: v["n_points"] for k, v in RESULTS.items()},
    "strategies": sorted({s for v in RESULTS.values() for s in v["strategy_counts"]}),
    "timestamp": time.time(),
}
json.dump(manifest, open(os.path.join(DRIVE, "manifest.json"), "w"), indent=2, ensure_ascii=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False)[:1200])
print("\n-> copy manifest.json into the Next.js repo; lib/manifest.ts asserts on it")

{
  "model": "jinaai/jina-embeddings-v5-text-small",
  "api_model": "jina-embeddings-v5-text-small",
  "revision": "dd76d535f5447ca3897a9c893fb1e612ead98192",
  "task": "retrieval",
  "doc_prompt": "document",
  "query_prompt": "query",
  "dim_primary": 256,
  "dim_rerank": 1024,
  "normalized": true,
  "truncation": "slice_then_normalize",
  "collection": "msmarco_xi",
  "dataset": "ai4bharat/MSMARCO-XI",
  "sparse": "Qdrant/bm25",
  "total_points": 53444,
  "languages": {
    "ta": 7195,
    "ml": 7483,
    "mr": 4000,
    "or": 2400,
    "bn": 2400
  },
  "strategies": [
    "atomic",
    "enriched",
    "query",
    "window",
    "xling"
  ],
  "timestamp": 1787406774.9761364
}

-> copy manifest.json into the Next.js repo; lib/manifest.ts asserts on it


In [15]:
client.count(COLLECTION, exact=True).count

53444

In [16]:
langs = ["hi","kn","ta","ml","mr","or","bn","en"]
for l in langs:
    n = client.count(COLLECTION, exact=True, count_filter=models.Filter(must=[
        models.FieldCondition(key="lang", match=models.MatchValue(value=l))])).count
    print(f"{l}: {n}")

hi: 16966
kn: 9110
ta: 6195
ml: 6483
mr: 4000
or: 2400
bn: 2400
en: 5890
